# Integração da base de alunos + Atlas + Censo Escolar com dados do Fundeb 2024

Este notebook dá continuidade ao processo de enriquecimento da base analítica
do projeto, incorporando informações relacionadas ao financiamento da educação
pública provenientes do **Fundo de Manutenção e Desenvolvimento da Educação
Básica e de Valorização dos Profissionais da Educação (Fundeb)**.

A base de alunos utilizada como ponto de partida já contém informações da
avaliação de alfabetização, indicadores socioeconômicos municipais derivados
do Atlas do Desenvolvimento Humano e indicadores educacionais construídos
a partir do Censo Escolar 2024.

Nesta etapa, será utilizada como fonte a publicação oficial do Fundeb referente
à **Receita Total do Fundeb por Ente Federado**, disponibilizada pelo Fundo
Nacional de Desenvolvimento da Educação (FNDE).

**Fonte oficial:**  
[FNDE — Receita Total do Fundeb por Ente Federado — 2024](https://www.gov.br/fnde/pt-br/acesso-a-informacao/acoes-e-programas/financiamento/fundeb/2024/copy3_of_ReceitaTotalporentefederado.pdf)

A fonte apresenta informações financeiras por ente federado, incluindo a
receita da contribuição de estados e municípios ao Fundeb, as complementações
da União nas modalidades VAAF, VAAT e VAAR, a complementação total da União
e o total das receitas previstas do Fundeb.

Como os dados são disponibilizados originalmente em formato PDF, será
necessário realizar sua extração e estruturação antes das etapas de análise,
auditoria e eventual integração à base de alunos.

O objetivo é investigar se informações relacionadas ao contexto de
financiamento educacional podem acrescentar uma nova dimensão analítica ao
projeto, complementando os contextos socioeconômico e educacional já
incorporados.

A inclusão de variáveis do Fundeb na base final não será definida previamente.
Os dados serão inicialmente extraídos, estruturados e auditados, e somente
depois serão avaliados os indicadores com potencial de contribuição para as
etapas posteriores de análise exploratória e modelagem.

## Variáveis disponíveis na fonte

A publicação utilizada apresenta informações de identificação do ente federado
e diferentes componentes da receita prevista do Fundeb.

As principais variáveis disponíveis são:

- **UF:** Unidade da Federação à qual pertence o ente federado.

- **Código IBGE:** código utilizado para identificação do ente federado.
  Na fonte, estão presentes tanto municípios quanto governos estaduais.

- **Ente federado:** identifica o município ou o Governo do Estado ao qual
  os valores apresentados se referem.

- **Receita da contribuição de estados e municípios ao Fundeb:** componente
  da receita do Fundeb proveniente da contribuição de estados e municípios,
  antes das complementações realizadas pela União.

- **Complementação VAAF (Valor Aluno Ano Fundeb):** modalidade de
  complementação da União relacionada ao alcance do valor mínimo nacional
  por aluno no âmbito dos Fundos estaduais.

- **Complementação VAAT (Valor Aluno Ano Total):** modalidade de
  complementação da União que considera os recursos vinculados à educação
  disponíveis para cada rede de ensino.

- **Complementação VAAR (Valor Aluno Ano por Resultado):** modalidade de
  complementação da União associada ao cumprimento de condicionalidades de
  gestão e à evolução de indicadores educacionais.

- **Complementação da União Total:** valor total das complementações da
  União atribuídas ao ente federado.

- **Total das receitas previstas:** total das receitas previstas do Fundeb
  atribuídas ao ente federado.

### Variáveis de interesse para o projeto

Embora todas as colunas sejam preservadas durante a extração da fonte,
nem todas possuem o mesmo interesse para o objetivo analítico deste projeto.

O interesse principal está nas variáveis que permitem representar de forma
mais geral o contexto de financiamento do Fundeb, especialmente:

- **Receita da contribuição de estados e municípios ao Fundeb**;
- **Complementação da União Total**;
- **Total das receitas previstas**.

A partir dessas informações, também poderá ser construído um indicador
relativo da participação da complementação da União na receita prevista
do Fundeb:

`proporção da complementação da União = Complementação da União Total / Total das receitas previstas`

O indicador será inicialmente tratado como candidato e sua permanência
nas etapas posteriores dependerá das análises e auditorias realizadas
ao longo do projeto.

## 1. Carregamento e estruturação da fonte Fundeb 2024

Nesta etapa, será realizada a leitura da fonte oficial do Fundeb 2024
contendo a receita total prevista por ente federado.

Diferentemente das fontes utilizadas anteriormente, os dados do Fundeb
selecionados para este projeto são disponibilizados originalmente em
formato PDF. Portanto, antes da análise dos dados, será necessário
extrair o conteúdo tabular do documento e estruturá-lo em um DataFrame.

A extração buscará preservar as informações apresentadas na fonte
original, incluindo a identificação do ente federado e os diferentes
componentes financeiros do Fundeb.

Antes de qualquer transformação definitiva, a estrutura obtida será
inspecionada e comparada com o documento original, permitindo avaliar
a qualidade da extração e identificar eventuais necessidades de
tratamento.

In [0]:
%pip install pdfplumber

In [0]:
# Objetivo:
#
# Extrair as tabelas do documento oficial
# do Fundeb 2024.
#
# Justificativa:
#
# A fonte selecionada é disponibilizada em
# formato PDF e contém informações financeiras
# organizadas em tabelas ao longo de suas páginas.
#
# A extração tabular permite preservar a estrutura
# de linhas e colunas do documento antes da
# aplicação de tratamentos e conversões.
#
# Nesta etapa, os dados serão mantidos em sua
# forma originalmente extraída para permitir
# auditorias posteriores.
#
# Ação:
#
# Percorre todas as páginas do PDF, extrai as
# tabelas identificadas e acumula seus registros
# para posterior estruturação em um DataFrame.

import pdfplumber
import pandas as pd

caminho_fundeb = (
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/fontes_externas/fundeb/"
    "ReceitaTotalporentefederado.pdf"
)

registros_fundeb = []

with pdfplumber.open(caminho_fundeb) as pdf:
    for pagina in pdf.pages:
        tabela = pagina.extract_table()

        if tabela is not None:
            registros_fundeb.extend(tabela)

In [0]:
# Objetivo:
#
# Validar a estrutura dos registros extraídos
# do PDF oficial do Fundeb 2024.
#
# Justificativa:
#
# Antes da construção do DataFrame, é necessário
# confirmar que a extração preservou a estrutura
# tabular esperada da fonte.
#
# O documento possui nove campos por registro,
# incluindo o cabeçalho da tabela.
#
# Ação:
#
# Verifica o total de registros extraídos, a
# quantidade de colunas observada e o número
# de cabeçalhos identificados.

pd.Series({
    "registros_extraidos": len(registros_fundeb),

    "quantidades_colunas_observadas": sorted({
        len(registro)
        for registro in registros_fundeb
    }),

    "cabecalhos_identificados": sum(
        registro[0] == "UF"
        for registro in registros_fundeb
    )
})

In [0]:
# Objetivo:
#
# Identificar os nomes dos campos presentes
# no cabeçalho extraído do PDF do Fundeb 2024.
#
# Justificativa:
#
# Embora a estrutura da tabela já tenha sido
# validada, os nomes das colunas devem ser
# observados exatamente como foram extraídos
# antes da seleção das variáveis de interesse.
#
# Isso evita assumir previamente grafias,
# quebras de linha ou outros caracteres
# introduzidos durante a leitura do PDF.
#
# Ação:
#
# Localiza o único registro identificado como
# cabeçalho e exibe seus nove campos.

cabecalho_fundeb = next(
    registro
    for registro in registros_fundeb
    if registro[0] == "UF"
)

cabecalho_fundeb

## 2. Seleção das variáveis de interesse

Após a validação estrutural dos registros extraídos, esta etapa tem como
objetivo selecionar apenas as variáveis do Fundeb 2024 consideradas
relevantes para o enriquecimento da base analítica do projeto.

Serão preservadas as informações de identificação geográfica e administrativa
do ente federado, além das variáveis financeiras que representam os recursos
previstos do Fundeb.

As variáveis selecionadas são:

- **UF**: unidade federativa;
- **Código IBGE**: código de identificação do ente federado;
- **Ente federado**: nome do estado ou município;
- **Receita da contribuição de estados e municípios ao Fundeb**;
- **Complementação da União Total**;
- **Total das receitas previstas**.

As colunas referentes às modalidades individuais de complementação
**VAAF**, **VAAT** e **VAAR** permanecem disponíveis nos registros originalmente
extraídos, mas não serão incorporadas à base analítica nesta etapa.

A seleção antecipada das variáveis reduz a complexidade da estrutura de
trabalho e mantém o processamento concentrado nas informações necessárias
para as etapas posteriores de preparação e integração dos dados.

In [0]:
# Objetivo:
#
# Selecionar, entre os campos extraídos do Fundeb
# 2024, apenas as variáveis de interesse para a
# construção da base analítica do projeto.
#
# Justificativa:
#
# A extração bruta preserva as nove colunas da
# fonte oficial. Entretanto, nesta etapa serão
# mantidas apenas as informações necessárias para
# identificação dos entes federados e as variáveis
# financeiras de interesse principal.
#
# A seleção é realizada pelas posições das colunas
# na estrutura original, preservando neste momento
# os nomes e os valores exatamente como extraídos
# do documento.
#
# Ação:
#
# Define as posições das seis variáveis de interesse
# e seleciona esses campos em cada registro, excluindo
# o cabeçalho da lista de dados.

indices_interesse = [0, 1, 2, 3, 7, 8]

registros_fundeb_selecionados = [
    [
        registro[indice]
        for indice in indices_interesse
    ]
    for registro in registros_fundeb
    if registro[0] != "UF"
]

len(registros_fundeb_selecionados)

In [0]:
# Objetivo:
#
# Validar a estrutura dos registros após a seleção
# das variáveis de interesse do Fundeb 2024.
#
# Justificativa:
#
# A etapa anterior reduziu a estrutura original de
# nove para seis campos por registro.
#
# Antes da construção do DataFrame, é necessário
# confirmar que essa seleção foi aplicada de forma
# consistente a todos os registros.
#
# Ação:
#
# Verifica o total de registros selecionados e as
# quantidades de campos observadas na nova estrutura.

pd.Series({
    "registros_selecionados": len(registros_fundeb_selecionados),

    "quantidades_colunas_observadas": sorted({
        len(registro)
        for registro in registros_fundeb_selecionados
    })
})

In [0]:
# Objetivo:
#
# Construir o DataFrame do Fundeb 2024 a partir
# dos registros e das variáveis selecionadas.
#
# Justificativa:
#
# A etapa anterior confirmou que os 5.595 registros
# de dados possuem exatamente os seis campos de
# interesse.
#
# Para manter a correspondência com a estrutura
# original da fonte, o cabeçalho será submetido à
# mesma seleção de posições aplicada aos registros.
#
# Neste momento, os nomes das colunas e os valores
# permanecem exatamente como foram extraídos do PDF.
#
# Ação:
#
# Seleciona os campos correspondentes no cabeçalho
# original e utiliza esses nomes para construir o
# DataFrame com os registros já selecionados.

cabecalho_fundeb_selecionado = [
    cabecalho_fundeb[indice]
    for indice in indices_interesse
]

df_fundeb = pd.DataFrame(
    registros_fundeb_selecionados,
    columns=cabecalho_fundeb_selecionado
)

df_fundeb.head()

In [0]:
# Objetivo:
#
# Padronizar os nomes das variáveis selecionadas
# do Fundeb 2024.
#
# Justificativa:
#
# Os nomes originais preservam a formatação do PDF,
# incluindo quebras de linha e descrições extensas.
#
# Para facilitar as etapas posteriores de análise,
# integração e modelagem, serão utilizados nomes
# curtos, descritivos e padronizados em snake_case.
#
# As variáveis financeiras recebem o sufixo
# "_fundeb" para preservar sua origem semântica
# após a integração com as demais fontes do projeto.
#
# Ação:
#
# Renomeia as seis colunas selecionadas, sem alterar
# os valores armazenados no DataFrame.

df_fundeb = df_fundeb.rename(columns={
    "UF": "uf",
    "Código\nIBGE": "codigo_ibge_municipio",
    "Ente federado": "municipio",
    "Receita da\ncontribuição de\nestados e municípios\nao Fundeb": "receita_contribuicao_fundeb",
    "Complementação da\nUnião Total": "complementacao_uniao_fundeb",
    "Total das receitas\nprevistas": "receita_total_fundeb"
})

df_fundeb.columns.tolist()

In [0]:
# Objetivo:
#
# Validar a estrutura final do DataFrame após a
# seleção e padronização das variáveis de interesse
# do Fundeb 2024.
#
# Justificativa:
#
# Antes de encerrar esta etapa, é importante
# confirmar que o DataFrame preserva os 5.595
# registros esperados e contém exclusivamente as
# seis variáveis selecionadas para o projeto.
#
# Ação:
#
# Verifica a quantidade de registros e colunas
# presentes no DataFrame resultante.

pd.Series({
    "registros": df_fundeb.shape[0],
    "colunas": df_fundeb.shape[1]
})

## 3. Auditoria e preparação dos dados do Fundeb

Após a seleção e padronização das variáveis de interesse, esta etapa tem como
objetivo avaliar a qualidade dos dados antes de sua integração à base analítica
do projeto.

A auditoria será realizada antes das transformações para que eventuais
inconsistências sejam identificadas ainda na representação obtida a partir da
fonte oficial.

Serão avaliados aspectos como:

- tipos de dados das variáveis;
- valores ausentes;
- duplicidade de registros;
- consistência do código IBGE dos municípios;
- granularidade dos registros;
- representação textual dos valores financeiros;
- possíveis padrões ou anomalias introduzidos durante a extração do PDF.

Somente após essa avaliação serão aplicadas as transformações necessárias
para preparar os dados do Fundeb para integração com as demais fontes do
projeto.

In [0]:
# Objetivo:
#
# Realizar o diagnóstico inicial das variáveis
# selecionadas do Fundeb 2024.
#
# Justificativa:
#
# Antes de aplicar qualquer transformação, é
# necessário compreender como o Pandas interpretou
# os dados extraídos do PDF e verificar a presença
# de valores ausentes.
#
# Esse diagnóstico permite identificar quais
# variáveis exigirão tratamento nas etapas
# seguintes sem modificar a representação atual
# dos dados.
#
# Ação:
#
# Exibe a estrutura do DataFrame, os tipos de dados
# identificados pelo Pandas e a quantidade de valores
# não nulos em cada variável.

df_fundeb.info()

In [0]:
# Objetivo:
#
# Verificar a existência de valores ausentes
# representados como strings vazias nas variáveis
# do Fundeb 2024.
#
# Justificativa:
#
# O diagnóstico inicial indicou 5.595 valores não
# nulos em todas as colunas. Entretanto, dados
# extraídos de PDF podem conter campos vazios
# representados como strings, que não são
# identificados pelo Pandas como valores nulos.
#
# Antes de concluir que a base não possui dados
# ausentes, é necessário verificar também esse
# tipo de representação.
#
# Ação:
#
# Remove apenas os espaços das extremidades para
# fins de comparação e contabiliza strings vazias
# em cada variável, sem modificar o DataFrame.

df_fundeb.apply(
    lambda coluna: coluna.str.strip().eq("").sum()
)

In [0]:
# Objetivo:
#
# Verificar a unicidade do código IBGE dos municípios
# na base selecionada do Fundeb 2024.
#
# Justificativa:
#
# A integração futura com a base analítica será
# realizada em granularidade municipal.
#
# Antes do merge, é necessário verificar se cada
# código IBGE identifica apenas um registro no
# DataFrame do Fundeb, evitando relações inesperadas
# de um-para-muitos durante a integração.
#
# Ação:
#
# Compara a quantidade total de registros com a
# quantidade de códigos IBGE municipais distintos
# e contabiliza códigos duplicados.

pd.Series({
    "registros": len(df_fundeb),

    "codigos_ibge_unicos": (
        df_fundeb["codigo_ibge_municipio"].nunique()
    ),

    "registros_com_codigo_duplicado": (
        df_fundeb["codigo_ibge_municipio"]
        .duplicated(keep=False)
        .sum()
    )
})

In [0]:
# Objetivo:
#
# Avaliar a distribuição geográfica dos registros
# do Fundeb 2024 por unidade federativa.
#
# Justificativa:
#
# A etapa anterior confirmou que os 5.595 códigos
# IBGE presentes na base são únicos.
#
# Antes de assumir que todos os registros possuem
# a granularidade municipal esperada para o merge,
# é importante observar como eles estão distribuídos
# entre as unidades federativas.
#
# Essa verificação também pode revelar registros
# adicionais ou padrões inesperados na estrutura
# extraída da fonte.
#
# Ação:
#
# Conta a quantidade de registros por UF e apresenta
# a distribuição em ordem alfabética.

df_fundeb["uf"].value_counts().sort_index()

In [0]:
# Objetivo:
#
# Avaliar a consistência estrutural dos códigos IBGE
# dos municípios presentes na base do Fundeb 2024.
#
# Justificativa:
#
# A unicidade dos códigos já foi confirmada, mas isso
# não garante que sua representação esteja adequada
# para utilização como chave de integração.
#
# Antes do confronto com as demais bases do projeto,
# é necessário verificar se os códigos possuem
# comprimento consistente e são formados apenas
# por caracteres numéricos.
#
# Ação:
#
# Identifica os comprimentos observados nos códigos
# IBGE e contabiliza registros que contenham algum
# caractere não numérico.

pd.Series({
    "comprimentos_observados": sorted(
        df_fundeb["codigo_ibge_municipio"]
        .str.len()
        .unique()
    ),

    "codigos_nao_numericos": (
        ~df_fundeb["codigo_ibge_municipio"]
        .str.isdigit()
    ).sum()
})

In [0]:
# Objetivo:
#
# Identificar os registros cujo código IBGE possui
# comprimento diferente do padrão municipal observado.
#
# Justificativa:
#
# A auditoria anterior revelou códigos com dois e
# sete dígitos.
#
# Como os códigos de sete dígitos são compatíveis
# com a estrutura utilizada para identificação
# municipal, os registros de dois dígitos precisam
# ser examinados antes de qualquer tratamento.
#
# Essa inspeção permitirá verificar se eles
# representam outra granularidade presente na
# fonte, como registros estaduais.
#
# Ação:
#
# Seleciona os registros cujo código IBGE não possui
# sete caracteres e exibe suas informações de
# identificação, sem modificar o DataFrame.

df_fundeb.loc[
    df_fundeb["codigo_ibge_municipio"].str.len() != 7,
    ["uf", "codigo_ibge_municipio", "municipio"]
]

In [0]:
# Objetivo:
#
# Construir uma base do Fundeb 2024 restrita aos
# registros de granularidade municipal.
#
# Justificativa:
#
# A auditoria identificou duas granularidades na
# fonte: registros municipais, representados por
# códigos IBGE de sete dígitos, e 27 registros dos
# governos estaduais e do Distrito Federal,
# representados por códigos de dois dígitos.
#
# Como a integração com a base analítica do projeto
# será realizada em nível municipal, será criada
# uma estrutura específica para os municípios,
# preservando o DataFrame anterior para auditoria.
#
# Ação:
#
# Seleciona os registros cujo código IBGE possui
# sete caracteres, cria uma cópia independente e
# reorganiza o índice do novo DataFrame.

df_fundeb_municipios = (
    df_fundeb[
        df_fundeb["codigo_ibge_municipio"].str.len() == 7
    ]
    .copy()
    .reset_index(drop=True)
)

df_fundeb_municipios.shape

In [0]:
# Objetivo:
#
# Investigar a presença de espaços internos nos
# valores financeiros do Fundeb 2024.
#
# Justificativa:
#
# A inspeção inicial revelou valores monetários
# extraídos do PDF com espaços inseridos entre
# algarismos, como "1 7.175.565,35".
#
# Antes de remover esses caracteres ou converter
# as variáveis para formato numérico, é necessário
# verificar a extensão desse padrão nas três
# variáveis financeiras.
#
# Ação:
#
# Conta, em cada variável financeira, quantos
# registros possuem pelo menos um espaço em sua
# representação textual, sem modificar os dados.

colunas_financeiras_fundeb = [
    "receita_contribuicao_fundeb",
    "complementacao_uniao_fundeb",
    "receita_total_fundeb"
]

df_fundeb_municipios[
    colunas_financeiras_fundeb
].apply(
    lambda coluna: coluna.str.contains(" ", regex=False).sum()
)

In [0]:
# Objetivo:
#
# Inspecionar exemplos dos valores financeiros que
# apresentam espaços em sua representação textual.
#
# Justificativa:
#
# A auditoria anterior mostrou que os espaços estão
# presentes em todos os registros das variáveis de
# receita da contribuição e receita total, além de
# parte dos registros de complementação da União.
#
# Antes de definir uma regra de limpeza, é necessário
# observar como esses espaços aparecem nos valores
# extraídos do PDF e verificar se o padrão é
# consistente entre as variáveis financeiras.
#
# Ação:
#
# Para cada variável financeira, seleciona os valores
# que contêm espaços e exibe algumas ocorrências
# distintas, sem modificar o DataFrame.

for coluna in colunas_financeiras_fundeb:

    exemplos = (
        df_fundeb_municipios.loc[
            df_fundeb_municipios[coluna].str.contains(" ", regex=False),
            coluna
        ]
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    print(f"\n{coluna}:")
    print(exemplos)

In [0]:
# Objetivo:
#
# Validar o padrão textual dos valores financeiros
# do Fundeb 2024 antes de sua transformação.
#
# Justificativa:
#
# A inspeção dos valores revelou espaços inseridos
# entre algarismos durante a extração do PDF.
#
# Antes de remover esses espaços e converter as
# variáveis para formato numérico, é necessário
# confirmar que, desconsiderando esse artefato,
# os valores seguem uma representação monetária
# consistente em toda a base municipal.
#
# Ação:
#
# Remove temporariamente os espaços apenas para
# fins de validação e verifica quantos valores não
# seguem o padrão monetário brasileiro esperado,
# sem modificar o DataFrame.

padrao_monetario = r"^\d{1,3}(?:\.\d{3})*,\d{2}$"

df_fundeb_municipios[
    colunas_financeiras_fundeb
].apply(
    lambda coluna: (
        ~coluna
        .str.replace(" ", "", regex=False)
        .str.match(padrao_monetario)
    ).sum()
)

In [0]:
# Objetivo:
#
# Identificar as representações presentes nos valores
# de complementação da União que não seguem o padrão
# monetário esperado.
#
# Justificativa:
#
# A validação anterior identificou 1.950 registros
# da variável de complementação da União que não
# apresentam formato monetário, mesmo após
# desconsiderar os espaços introduzidos na extração.
#
# A consulta à fonte oficial indicou que o caractere
# "-" é utilizado quando não há valor de
# complementação da União para o ente federado.
#
# Antes de definir qualquer regra de transformação,
# é necessário confirmar se os registros fora do
# padrão presentes na base extraída correspondem
# exclusivamente a essa representação.
#
# Ação:
#
# Seleciona os valores que não seguem o padrão
# monetário, remove duplicidades e apresenta as
# representações distintas encontradas, sem
# modificar o DataFrame.

valores_fora_padrao_complementacao = (
    df_fundeb_municipios.loc[
        ~df_fundeb_municipios["complementacao_uniao_fundeb"]
        .str.replace(" ", "", regex=False)
        .str.match(padrao_monetario),
        "complementacao_uniao_fundeb"
    ]
    .value_counts()
)

valores_fora_padrao_complementacao

In [0]:
# Objetivo:
#
# Preparar as variáveis financeiras do Fundeb 2024
# para utilização em análises e integrações posteriores.
#
# Justificativa:
#
# A auditoria identificou espaços introduzidos durante
# a extração do PDF nos valores monetários.
#
# Também foi confirmado que a representação "-" ocorre
# exclusivamente na variável de complementação da União
# e indica ausência de valor de complementação, sendo
# portanto representada numericamente por zero.
#
# Os valores monetários utilizam ainda a formatação
# brasileira, com ponto como separador de milhares e
# vírgula como separador decimal.
#
# A estrutura municipal anterior será preservada,
# criando-se uma cópia independente para aplicação
# dos tratamentos.
#
# Ação:
#
# Cria uma cópia da base municipal, remove os artefatos
# de espaço, representa a ausência de complementação
# por zero, converte a notação monetária brasileira e
# transforma as três variáveis financeiras em float.

df_fundeb_municipios_tratado = df_fundeb_municipios.copy()

df_fundeb_municipios_tratado[colunas_financeiras_fundeb] = (
    df_fundeb_municipios_tratado[colunas_financeiras_fundeb]
    .apply(
        lambda coluna: (
            coluna
            .str.replace(" ", "", regex=False)
            .replace("-", "0")
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
            .astype(float)
        )
    )
)

df_fundeb_municipios_tratado[
    colunas_financeiras_fundeb
].head()

In [0]:
# Objetivo:
#
# Validar a consistência financeira dos valores
# tratados do Fundeb 2024.
#
# Justificativa:
#
# Após a remoção dos espaços introduzidos na
# extração, a conversão da notação monetária
# brasileira e a representação de "-" como zero,
# é necessário verificar se as transformações
# preservaram a relação entre os componentes
# financeiros da fonte.
#
# A receita total prevista deve corresponder à soma
# da receita da contribuição ao Fundeb com a
# complementação total da União.
#
# Como os valores estão representados em ponto
# flutuante, a comparação utiliza uma pequena
# tolerância numérica.
#
# Ação:
#
# Compara a receita total informada com a soma de
# seus componentes e contabiliza quantos registros
# apresentam ou não consistência financeira.

import numpy as np

receita_total_calculada = (
    df_fundeb_municipios_tratado["receita_contribuicao_fundeb"]
    + df_fundeb_municipios_tratado["complementacao_uniao_fundeb"]
)

valores_consistentes = np.isclose(
    receita_total_calculada,
    df_fundeb_municipios_tratado["receita_total_fundeb"],
    atol=0.01
)

pd.Series({
    "registros_avaliados": len(df_fundeb_municipios_tratado),
    "registros_consistentes": valores_consistentes.sum(),
    "registros_inconsistentes": (~valores_consistentes).sum()
})

In [0]:
# Objetivo:
#
# Validar a estrutura e os tipos de dados da base
# municipal do Fundeb 2024 após o tratamento das
# variáveis financeiras.
#
# Justificativa:
#
# A consistência financeira dos 5.568 registros já
# foi confirmada. Antes de avançar para a integração
# com a base analítica do projeto, é necessário
# verificar se as variáveis financeiras foram
# efetivamente convertidas para tipos numéricos e
# se a estrutura esperada foi preservada.
#
# Ação:
#
# Exibe a estrutura do DataFrame tratado, os tipos
# de dados e a quantidade de valores não nulos em
# cada variável.

df_fundeb_municipios_tratado.info()

## 4. Auditoria pré-integração

Após a auditoria e preparação dos dados do Fundeb 2024, esta etapa tem como
objetivo avaliar sua integração à base analítica construída nas etapas
anteriores do projeto.

A base municipal tratada contém 5.568 registros e utiliza o código IBGE do
município como chave de identificação geográfica. Antes da realização do
merge, será verificada a compatibilidade dessa chave com a base analítica,
bem como a cobertura dos municípios entre as duas fontes.

Essa validação é necessária para identificar previamente possíveis municípios
sem correspondência e evitar perdas ou multiplicações inesperadas de registros
durante a integração.

Após a análise das chaves, a integração será realizada preservando a
granularidade da base analítica e serão aplicadas validações pós-merge para
confirmar a integridade do resultado.

In [0]:
# Objetivo:
#
# Carregar a base analítica construída nas etapas
# anteriores do projeto.
#
# Justificativa:
#
# A integração dos dados do Fundeb será realizada
# sobre a base já enriquecida com informações do
# Atlas do Desenvolvimento Humano e do Censo Escolar.
#
# A leitura do arquivo persistido garante que esta
# etapa utilize efetivamente o resultado consolidado
# dos notebooks anteriores, preservando a
# rastreabilidade do pipeline.
#
# Ação:
#
# Carrega o arquivo CSV da base analítica consolidada
# para utilização nas etapas de validação das chaves
# e posterior integração com os dados do Fundeb.


df_alunos_atlas_censo = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo/alunos_atlas_censo.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_alunos_atlas_censo.shape

In [0]:
# Objetivo:
#
# Avaliar a chave municipal da base analítica
# construída nas etapas anteriores do projeto.
#
# Justificativa:
#
# A integração com os dados do Fundeb será realizada
# por meio do código IBGE do município.
#
# Na base analítica consolidada, essa informação está
# representada pela variável CO_MUNICIPIO, enquanto
# na base do Fundeb a chave correspondente é
# codigo_ibge_municipio.
#
# Antes de comparar as duas fontes, é necessário
# verificar a representação atual da chave na base
# receptora, incluindo seu tipo, valores ausentes e
# quantidade de municípios distintos.
#
# Ação:
#
# Verifica o tipo da variável CO_MUNICIPIO,
# contabiliza valores ausentes e identifica a
# quantidade de códigos municipais distintos.

pd.Series({
    "dtype": str(
        df_alunos_atlas_censo["CO_MUNICIPIO"].dtype
    ),

    "valores_ausentes": (
        df_alunos_atlas_censo["CO_MUNICIPIO"]
        .isna()
        .sum()
    ),

    "codigos_municipais_distintos": (
        df_alunos_atlas_censo["CO_MUNICIPIO"]
        .nunique()
    )
})

In [0]:
# Objetivo:
#
# Avaliar a cobertura dos códigos municipais entre
# a base analítica e os dados do Fundeb 2024.
#
# Justificativa:
#
# A base analítica possui 5.556 municípios distintos
# com CO_MUNICIPIO informado, além de 510 registros
# sem chave municipal, situação previamente conhecida
# e documentada.
#
# A base tratada do Fundeb possui 5.568 municípios
# distintos, identificados por codigo_ibge_municipio.
#
# Antes da integração, é necessário comparar os
# conjuntos de códigos municipais para identificar
# possíveis municípios sem correspondência entre
# as duas fontes.
#
# Ação:
#
# Constrói conjuntos comparáveis de códigos municipais,
# desconsiderando os registros sem chave na base
# analítica, e contabiliza as diferenças de cobertura
# entre as duas fontes.

municipios_alunos = set(
    df_alunos_atlas_censo["CO_MUNICIPIO"]
    .dropna()
    .astype("int64")
)

municipios_fundeb = set(
    df_fundeb_municipios_tratado["codigo_ibge_municipio"]
    .astype("int64")
)

pd.Series({
    "municipios_alunos": len(municipios_alunos),

    "municipios_fundeb": len(municipios_fundeb),

    "alunos_sem_correspondencia_fundeb": len(
        municipios_alunos - municipios_fundeb
    ),

    "fundeb_sem_correspondencia_alunos": len(
        municipios_fundeb - municipios_alunos
    )
})

In [0]:
# Objetivo:
#
# Identificar os municípios presentes na base
# analítica que não possuem correspondência nos
# dados do Fundeb 2024.
#
# Justificativa:
#
# A comparação das chaves municipais identificou
# três municípios representados na base de alunos
# que não estão presentes na base tratada do Fundeb.
#
# Antes da integração, é necessário identificar
# esses municípios e quantificar os alunos associados
# a cada um deles, permitindo antecipar e documentar
# as ausências esperadas após o merge.
#
# Ação:
#
# Seleciona os registros pertencentes aos códigos
# municipais sem correspondência no Fundeb, agrupa
# por código, município e UF e contabiliza a
# quantidade de alunos associada a cada município.

municipios_sem_fundeb = (
    df_alunos_atlas_censo[
        df_alunos_atlas_censo["CO_MUNICIPIO"]
        .isin(municipios_alunos - municipios_fundeb)
    ]
    .groupby(
        ["CO_MUNICIPIO", "NO_MUNICIPIO", "SG_UF"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "quantidade_alunos"})
    .sort_values(
        "quantidade_alunos",
        ascending=False
    )
)

municipios_sem_fundeb

In [0]:
# Objetivo:
#
# Investigar a representação do Distrito Federal
# nos registros originalmente extraídos do Fundeb 2024.
#
# Justificativa:
#
# A comparação das chaves identificou Brasília,
# código IBGE 5300108, entre os municípios presentes
# na base analítica sem correspondência na base
# municipal tratada do Fundeb.
#
# Durante a auditoria da fonte, foi identificado um
# registro do Distrito Federal classificado como
# "GOVERNO DO ESTADO", posteriormente excluído junto
# aos demais registros estaduais.
#
# Antes de definir qualquer tratamento específico,
# é necessário recuperar esse registro na estrutura
# original e verificar como o Distrito Federal está
# representado na fonte.
#
# Ação:
#
# Seleciona na base anterior à exclusão dos registros
# estaduais todas as ocorrências referentes ao DF,
# permitindo inspecionar sua identificação e seus
# valores financeiros sem modificar os dados.

df_fundeb.loc[
    df_fundeb["uf"] == "DF"
]

In [0]:
# Objetivo:
#
# Investigar a representação de Fernando de Noronha
# nos dados extraídos do Fundeb 2024.
#
# Justificativa:
#
# A comparação das chaves identificou Fernando de
# Noronha, código IBGE 2605459, entre os municípios
# presentes na base analítica sem correspondência na
# base municipal tratada do Fundeb.
#
# Antes de interpretar essa ausência como diferença
# de cobertura ou particularidade administrativa,
# é necessário verificar se Fernando de Noronha
# aparece em algum registro anterior à exclusão dos
# governos estaduais.
#
# Ação:
#
# Pesquisa na base do Fundeb registros que possuam
# o código 2605459 ou referência textual a Noronha,
# sem modificar o DataFrame.

df_fundeb.loc[
    (
        df_fundeb["codigo_ibge_municipio"]
        .eq("2605459")
    )
    |
    (
        df_fundeb["municipio"]
        .str.contains(
            "NORONHA",
            case=False,
            na=False
        )
    )
]

In [0]:
# Objetivo:
#
# Harmonizar a representação territorial do Distrito
# Federal para permitir sua integração à base analítica.
#
# Justificativa:
#
# A fonte do Fundeb representa o Distrito Federal como
# ente distrital, utilizando o código "53" e a descrição
# "GOVERNO DO ESTADO".
#
# Na base analítica, os alunos do Distrito Federal estão
# associados a Brasília pelo código IBGE municipal 5300108.
#
# A auditoria das fontes confirmou que essa diferença
# decorre da forma de representação administrativa entre
# as bases, sendo necessária uma regra explícita de
# harmonização para a integração.
#
# Para preservar a rastreabilidade, os DataFrames
# originais não serão modificados.
#
# Ação:
#
# Recupera exclusivamente o registro do Distrito Federal
# na base anterior à exclusão dos governos estaduais,
# cria uma cópia, aplica o mesmo tratamento monetário já
# validado e harmoniza sua identificação para a chave
# municipal utilizada na base analítica.

df_fundeb_df_harmonizado = (
    df_fundeb.loc[
        (df_fundeb["uf"] == "DF")
        & (df_fundeb["codigo_ibge_municipio"] == "53")
    ]
    .copy()
)

df_fundeb_df_harmonizado[colunas_financeiras_fundeb] = (
    df_fundeb_df_harmonizado[colunas_financeiras_fundeb]
    .apply(
        lambda coluna: (
            coluna
            .str.replace(" ", "", regex=False)
            .replace("-", "0")
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
            .astype(float)
        )
    )
)

df_fundeb_df_harmonizado["codigo_ibge_municipio"] = "5300108"
df_fundeb_df_harmonizado["municipio"] = "BRASILIA"

df_fundeb_df_harmonizado

In [0]:
# Objetivo:
#
# Construir a base do Fundeb preparada para integração
# com a base analítica do projeto.
#
# Justificativa:
#
# A base municipal tratada possui 5.568 registros,
# mas não contém Brasília porque o Distrito Federal
# é representado na fonte do Fundeb como ente distrital,
# sob o código "53".
#
# Após a investigação e harmonização dessa exceção,
# foi criado um registro correspondente a Brasília,
# utilizando o código IBGE municipal 5300108 e
# preservando os valores financeiros do Distrito Federal.
#
# Para manter a rastreabilidade do tratamento, a base
# municipal anteriormente validada será preservada e
# uma nova versão será criada para a integração.
#
# Ação:
#
# Cria uma cópia da base municipal tratada e acrescenta
# exclusivamente o registro harmonizado do Distrito
# Federal, reorganizando o índice do DataFrame resultante.

df_fundeb_integracao = pd.concat(
    [
        df_fundeb_municipios_tratado.copy(),
        df_fundeb_df_harmonizado.copy()
    ],
    ignore_index=True
)

df_fundeb_integracao.shape

In [0]:
# Objetivo:
#
# Validar a unicidade da chave municipal na base do
# Fundeb preparada para integração.
#
# Justificativa:
#
# A base municipal tratada possuía 5.568 códigos IBGE
# únicos. Após a inclusão do registro harmonizado do
# Distrito Federal, a base destinada à integração passou
# a conter 5.569 registros.
#
# Antes de reavaliar a cobertura em relação à base
# analítica, é necessário confirmar que a harmonização
# não introduziu duplicidades na chave municipal.
#
# Ação:
#
# Verifica a quantidade de registros, o número de códigos
# IBGE distintos e a existência de códigos duplicados na
# base preparada para integração.

pd.Series({
    "registros": len(df_fundeb_integracao),

    "codigos_ibge_unicos": (
        df_fundeb_integracao["codigo_ibge_municipio"]
        .nunique()
    ),

    "registros_com_codigo_duplicado": (
        df_fundeb_integracao["codigo_ibge_municipio"]
        .duplicated()
        .sum()
    )
})

In [0]:
# Objetivo:
#
# Reavaliar a cobertura dos códigos municipais entre
# a base analítica e os dados do Fundeb após a
# harmonização do Distrito Federal.
#
# Justificativa:
#
# A comparação anterior identificou três municípios
# da base analítica sem correspondência no Fundeb.
#
# A investigação demonstrou que Brasília utiliza o
# código IBGE municipal 5300108 na base analítica,
# enquanto o Distrito Federal é representado de forma
# distinta na fonte do Fundeb.
#
# Após a harmonização explícita dessa representação,
# a base preparada para integração passou a conter
# 5.569 códigos IBGE únicos.
#
# Antes do merge, é necessário verificar o efeito
# dessa harmonização sobre a cobertura entre as fontes.
#
# Ação:
#
# Reconstrói o conjunto de códigos municipais do Fundeb
# a partir da base preparada para integração e compara
# novamente sua cobertura com os municípios presentes
# na base analítica.

municipios_fundeb_integracao = set(
    df_fundeb_integracao["codigo_ibge_municipio"]
    .astype("int64")
)

pd.Series({
    "municipios_alunos": len(municipios_alunos),

    "municipios_fundeb": len(
        municipios_fundeb_integracao
    ),

    "municipios_em_comum": len(
        municipios_alunos
        & municipios_fundeb_integracao
    ),

    "alunos_sem_correspondencia_fundeb": len(
        municipios_alunos
        - municipios_fundeb_integracao
    ),

    "fundeb_sem_correspondencia_alunos": len(
        municipios_fundeb_integracao
        - municipios_alunos
    )
})

In [0]:
# Objetivo:
#
# Identificar os municípios presentes na base do Fundeb
# preparada para integração que não estão representados
# na base analítica de alunos.
#
# Justificativa:
#
# A auditoria de cobertura identificou 15 códigos
# municipais presentes no Fundeb que não aparecem entre
# os 5.556 municípios representados na base analítica.
#
# Como a integração será realizada por meio de um left
# merge a partir da base de alunos, esses registros não
# serão incorporados ao resultado e não representam,
# por si só, um problema para a integração.
#
# Ainda assim, sua identificação permite documentar
# integralmente as diferenças de cobertura entre as
# duas fontes antes da realização do merge.
#
# Ação:
#
# Seleciona na base do Fundeb os registros cujos códigos
# IBGE não pertencem ao conjunto de municípios presentes
# na base analítica e exibe suas informações de
# identificação territorial.

fundeb_sem_alunos = (
    df_fundeb_integracao.loc[
        ~df_fundeb_integracao["codigo_ibge_municipio"]
        .astype("int64")
        .isin(municipios_alunos),
        [
            "uf",
            "codigo_ibge_municipio",
            "municipio"
        ]
    ]
    .sort_values(
        ["uf", "municipio"]
    )
    .reset_index(drop=True)
)

fundeb_sem_alunos

## 5. Integração do Fundeb à base analítica

Após a preparação dos dados do Fundeb e a auditoria de compatibilidade
entre as chaves municipais, esta etapa tem como objetivo incorporar as
variáveis financeiras à base analítica construída nas etapas anteriores
do projeto.

A análise prévia confirmou que a base preparada do Fundeb possui um único
registro por código municipal, permitindo uma relação de integração
muitos-para-um (`many_to_one`) com a base de alunos.

A comparação entre as fontes identificou 5.554 municípios em comum.
Também foram documentadas previamente as situações que não possuem
correspondência direta no Fundeb:

- 510 alunos sem `CO_MUNICIPIO` informado na base de origem;
- 110 alunos de Boa Esperança do Norte (MT), município de criação recente;
- 41 alunos de Fernando de Noronha (PE), sem registro individual na fonte
  do Fundeb utilizada.

O Distrito Federal foi tratado separadamente durante a preparação dos
dados, harmonizando sua representação na fonte do Fundeb com o código
IBGE de Brasília (`5300108`) utilizado na base analítica.

Dessa forma, espera-se que 661 registros permaneçam sem correspondência
após a integração.

O merge será realizado preservando integralmente a população da base
analítica, utilizando uma junção à esquerda (`left merge`), validação de
cardinalidade `many_to_one` e indicador de correspondência para permitir
a auditoria posterior do resultado.

Após a integração, serão verificadas a preservação da quantidade de
registros, a correspondência entre as bases, a origem dos casos sem
match e a completude das novas variáveis financeiras.

In [0]:
# Objetivo:
#
# Integrar os dados financeiros do Fundeb 2024 à
# base analítica construída nas etapas anteriores.
#
# Justificativa:
#
# A auditoria prévia confirmou que a base preparada
# do Fundeb possui um único registro por código
# municipal e que a integração apresenta cardinalidade
# muitos-para-um em relação à base de alunos.
#
# As chaves possuem nomes e tipos distintos entre as
# fontes. Na base analítica, CO_MUNICIPIO está
# representado como float64 devido à presença de
# valores ausentes, enquanto codigo_ibge_municipio
# está armazenado como string na base do Fundeb.
#
# Antes do merge, será criada uma cópia da base do
# Fundeb com a chave convertida para o mesmo domínio
# numérico da base analítica, preservando os
# DataFrames anteriormente validados.
#
# A integração será realizada por meio de left merge,
# mantendo integralmente os registros da base analítica.
# A cardinalidade será validada como many_to_one e
# será criada uma coluna auxiliar para posterior
# auditoria da correspondência entre as fontes.
#
# Ação:
#
# Prepara uma cópia da base do Fundeb, harmoniza o
# tipo da chave municipal e realiza a integração com
# validação de cardinalidade e indicador de merge.

df_fundeb_merge = df_fundeb_integracao.copy()

df_fundeb_merge["codigo_ibge_municipio"] = (
    df_fundeb_merge["codigo_ibge_municipio"]
    .astype("int64")
)

df_alunos_atlas_censo_fundeb = (
    df_alunos_atlas_censo.merge(
        df_fundeb_merge,
        how="left",
        left_on="CO_MUNICIPIO",
        right_on="codigo_ibge_municipio",
        validate="many_to_one",
        indicator=True
    )
)

df_alunos_atlas_censo_fundeb.shape

In [0]:
# Objetivo:
#
# Auditar a correspondência entre a base analítica
# e os dados financeiros do Fundeb 2024 após o merge.
#
# Justificativa:
#
# A integração foi realizada com indicator=True,
# criando a coluna auxiliar _merge, que permite
# identificar quais registros encontraram
# correspondência na base do Fundeb.
#
# A auditoria prévia das chaves identificou três
# situações que podem resultar em ausência de
# correspondência:
#
# - 510 alunos sem CO_MUNICIPIO informado;
# - 110 alunos de Boa Esperança do Norte (MT);
# - 41 alunos de Fernando de Noronha (PE).
#
# Dessa forma, são esperados 661 registros sem
# correspondência e 1.965.944 registros com
# correspondência no Fundeb.
#
# Ação:
#
# Contabiliza os registros segundo a situação de
# correspondência registrada na coluna _merge.

df_alunos_atlas_censo_fundeb["_merge"].value_counts()

In [0]:
# Objetivo:
#
# Reconciliar os registros sem correspondência no
# Fundeb 2024 segundo as causas identificadas antes
# da integração.
#
# Justificativa:
#
# A auditoria do merge identificou 661 registros
# classificados como left_only.
#
# Antes da integração, haviam sido identificados:
#
# - 510 alunos sem CO_MUNICIPIO informado;
# - 110 alunos de Boa Esperança do Norte (MT),
#   município sem correspondência direta no Fundeb 2024;
# - 41 alunos de Fernando de Noronha (PE), município
#   sem registro individual na fonte do Fundeb utilizada.
#
# A reconciliação permite confirmar que todos os casos
# sem correspondência são explicados exclusivamente
# por essas situações previamente conhecidas.
#
# Ação:
#
# Isola os registros classificados como left_only e
# contabiliza separadamente os alunos sem código
# municipal, os alunos de Boa Esperança do Norte,
# os alunos de Fernando de Noronha e quaisquer
# outros casos eventualmente encontrados.

left_only = df_alunos_atlas_censo_fundeb[
    df_alunos_atlas_censo_fundeb["_merge"] == "left_only"
]

pd.Series({
    "sem_codigo_municipal": (
        left_only["CO_MUNICIPIO"].isna().sum()
    ),

    "boa_esperanca_do_norte": (
        left_only["CO_MUNICIPIO"].eq(5101837).sum()
    ),

    "fernando_de_noronha": (
        left_only["CO_MUNICIPIO"].eq(2605459).sum()
    ),

    "outros_casos": (
        (
            left_only["CO_MUNICIPIO"].notna()
            & ~left_only["CO_MUNICIPIO"].isin([
                5101837,
                2605459
            ])
        ).sum()
    )
})

In [0]:
# Objetivo:
#
# Validar a completude das variáveis financeiras do
# Fundeb após sua integração à base analítica.
#
# Justificativa:
#
# A auditoria do merge identificou 661 registros sem
# correspondência no Fundeb, todos reconciliados com
# situações previamente conhecidas:
#
# - 510 alunos sem CO_MUNICIPIO informado;
# - 110 alunos de Boa Esperança do Norte (MT);
# - 41 alunos de Fernando de Noronha (PE).
#
# Antes da integração, as três variáveis financeiras
# estavam completas na base preparada do Fundeb.
#
# Dessa forma, espera-se que os únicos valores ausentes
# após o merge sejam aqueles associados aos 661
# registros classificados como left_only.
#
# Ação:
#
# Contabiliza os valores ausentes em cada variável
# financeira incorporada pelo Fundeb, compara o
# resultado com o total esperado e verifica a
# conformidade entre os valores observados e esperados.

variaveis_fundeb = [
    "receita_contribuicao_fundeb",
    "complementacao_uniao_fundeb",
    "receita_total_fundeb"
]

total_esperado_sem_fundeb = 661

ausencias_variaveis_fundeb = pd.DataFrame({
    "ausentes": (
        df_alunos_atlas_censo_fundeb[variaveis_fundeb]
        .isna()
        .sum()
    )
})

ausencias_variaveis_fundeb["esperado"] = (
    total_esperado_sem_fundeb
)

ausencias_variaveis_fundeb["conforme"] = (
    ausencias_variaveis_fundeb["ausentes"]
    == ausencias_variaveis_fundeb["esperado"]
)

ausencias_variaveis_fundeb

In [0]:
# Objetivo:
#
# Auditar as colunas resultantes da integração com
# os dados do Fundeb 2024.
#
# Justificativa:
#
# A base analítica possuía 36 colunas antes da
# integração e passou a possuir 43 após o merge.
#
# Além das três variáveis financeiras de interesse,
# a tabela do Fundeb contém variáveis auxiliares de
# identificação territorial e a integração criou a
# coluna temporária _merge.
#
# Antes de remover qualquer variável, é necessário
# identificar exatamente quais colunas foram
# acrescentadas à base analítica.
#
# Ação:
#
# Compara os nomes das colunas existentes antes e
# depois do merge e apresenta somente aquelas
# acrescentadas durante a integração com o Fundeb.

colunas_adicionadas_fundeb = [
    coluna
    for coluna in df_alunos_atlas_censo_fundeb.columns
    if coluna not in df_alunos_atlas_censo.columns
]

colunas_adicionadas_fundeb

In [0]:
# Objetivo:
#
# Remover as colunas auxiliares utilizadas durante
# a integração com os dados do Fundeb 2024.
#
# Justificativa:
#
# As variáveis uf, codigo_ibge_municipio e municipio
# foram incorporadas a partir da tabela do Fundeb
# apenas como informações auxiliares de identificação
# territorial.
#
# A base analítica já possui suas próprias variáveis
# de identificação municipal, incluindo CO_MUNICIPIO,
# utilizado como chave principal da integração.
#
# A coluna _merge foi criada temporariamente pelo
# parâmetro indicator=True e já cumpriu sua finalidade
# após a auditoria e reconciliação dos 661 registros
# sem correspondência.
#
# Dessa forma, somente as três variáveis financeiras
# do Fundeb devem permanecer como novos atributos
# analíticos na base resultante.
#
# Ação:
#
# Remove as variáveis auxiliares provenientes do Fundeb
# e a coluna de auditoria do merge e verifica as
# dimensões da base resultante.

df_alunos_atlas_censo_fundeb = (
    df_alunos_atlas_censo_fundeb.drop(
        columns=[
            "uf",
            "codigo_ibge_municipio",
            "municipio",
            "_merge"
        ]
    )
)

df_alunos_atlas_censo_fundeb.shape

In [0]:
# Objetivo:
#
# Verificar a estrutura final da base analítica após
# a integração dos dados financeiros do Fundeb 2024.
#
# Justificativa:
#
# Após o merge, a auditoria das correspondências e a
# remoção das colunas auxiliares, a base resultante
# possui 1.966.605 registros e 39 variáveis.
#
# Antes de encerrar a etapa de integração, é necessário
# verificar os tipos de dados, a quantidade de valores
# não nulos e a estrutura geral do DataFrame resultante.
#
# Ação:
#
# Exibe as informações estruturais da base analítica
# após a incorporação das três variáveis financeiras
# do Fundeb.

df_alunos_atlas_censo_fundeb.info()

In [0]:
# Objetivo:
#
# Realizar a validação estrutural final da base
# analítica após a integração do Fundeb 2024.
#
# Justificativa:
#
# As etapas anteriores confirmaram a preservação dos
# registros, a cardinalidade do merge, a origem dos
# casos sem correspondência e a completude das três
# variáveis financeiras incorporadas.
#
# Como verificação final, será confirmada a dimensão
# da base resultante e a presença das três variáveis
# selecionadas do Fundeb.
#
# Ação:
#
# Verifica a quantidade final de registros e colunas
# e confirma se todas as variáveis financeiras do
# Fundeb estão presentes na base analítica.

pd.Series({
    "registros": len(df_alunos_atlas_censo_fundeb),

    "colunas": df_alunos_atlas_censo_fundeb.shape[1],

    "variaveis_fundeb_presentes": all(
        coluna in df_alunos_atlas_censo_fundeb.columns
        for coluna in variaveis_fundeb
    )
})

### Resultado da integração com o Fundeb

A integração dos dados financeiros do Fundeb 2024 à base analítica foi
concluída e validada.

O `left merge` preservou integralmente os **1.966.605 registros** da base
de alunos e respeitou a cardinalidade `many_to_one`, previamente
verificada pela unicidade da chave municipal na base do Fundeb.

Dos registros da base analítica, **1.965.944 encontraram correspondência**
com os dados do Fundeb e **661 permaneceram sem correspondência**. Esses
casos foram integralmente reconciliados com as situações identificadas
antes da integração:

- **510 alunos** sem `CO_MUNICIPIO` informado;
- **110 alunos** de Boa Esperança do Norte (MT);
- **41 alunos** de Fernando de Noronha (PE).

Não foram identificados outros casos sem correspondência.

A representação do Distrito Federal foi previamente harmonizada com o
código IBGE de Brasília (`5300108`), permitindo sua integração à base
analítica sem alterar os registros municipais originalmente preparados.

As três variáveis financeiras incorporadas apresentaram exatamente
**661 valores ausentes**, correspondentes aos registros sem associação
ao Fundeb, sem ocorrência de ausências adicionais:

- `receita_contribuicao_fundeb`;
- `complementacao_uniao_fundeb`;
- `receita_total_fundeb`.

Após a conclusão das auditorias, as variáveis auxiliares utilizadas no
processo de integração foram removidas.

A base resultante possui **1.966.605 registros e 39 variáveis**, mantendo
as 36 variáveis existentes anteriormente e acrescentando exclusivamente
as três variáveis financeiras selecionadas do Fundeb.

Dessa forma, a integração foi considerada consistente e a base está
preparada para as etapas seguintes do projeto.

## 6. Persistência da base analítica enriquecida

Após a integração e validação dos dados do Fundeb 2024, a base analítica
resultante encontra-se consolidada com informações provenientes da camada
Gold da Fase 2 e dos enriquecimentos realizados com o Atlas do
Desenvolvimento Humano, o Censo Escolar e o Fundeb.

A base final desta etapa possui **1.966.605 registros e 39 variáveis**,
preservando integralmente a população de alunos e incorporando as três
variáveis financeiras selecionadas do Fundeb.

Nesta etapa, o objetivo é persistir a versão consolidada da base para que
ela possa ser utilizada nas fases seguintes de análise exploratória,
preparação para modelagem e construção dos modelos supervisionados.

A persistência será realizada em formato CSV, mantendo o padrão de
codificação e separação utilizado nos arquivos produzidos anteriormente
no projeto.

Após a gravação, o arquivo será novamente carregado e auditado para
confirmar que sua estrutura foi preservada durante o processo de
persistência.

In [0]:
# Objetivo:
#
# Persistir a base final enriquecida após a
# integração com o Fundeb 2024.
#
# Justificativa:
#
# A base resultante foi validada quanto à
# cardinalidade, correspondência entre as fontes
# e completude das variáveis incorporadas.
#
# A persistência permite disponibilizar a base
# consolidada para utilização nas próximas etapas
# do projeto.
#
# Ação:
#
# Salva a base final em formato CSV na camada Gold,
# utilizando o padrão de separador e codificação
# adotado no projeto.

df_alunos_atlas_censo_fundeb.to_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo_fundeb/alunos_atlas_censo_fundeb.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

In [0]:
# Objetivo:
#
# Validar o arquivo persistido após a
# integração com o Fundeb 2024.
#
# Justificativa:
#
# A releitura do arquivo permite confirmar
# que a base foi gravada corretamente e
# preservou sua dimensão antes de ser utilizada
# nas próximas etapas do projeto.
#
# Ação:
#
# Realiza uma leitura de controle do arquivo
# persistido e verifica sua dimensão.

df_validacao = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo_fundeb/alunos_atlas_censo_fundeb.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_validacao.shape